In [1]:
import openeo
from pathlib import Path

In [2]:
connection_cdse = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [3]:
# cheating:
res = ["s3://EODATA/Sentinel-2/MSI/L1C/2024/11/13/S2A_MSIL1C_20241113T101251_N0511_R022_T32TPQ_20241113T121135.SAFE"]

In [4]:
udf_code = f"""
from openeo.udf.udf_data import UdfData
from openeo.udf.structured_data import StructuredData

def apply_udf_data(data: UdfData):
    data.set_structured_data_list([
        StructuredData({res})
    ])
"""
udf = openeo.UDF(udf_code)
udf_process_graph = connection_cdse.datacube_from_process(
    "run_udf",
    data=[],
    udf=udf_code,
    context={},
    runtime="Python",
)

print(udf_process_graph.to_json())

{
  "process_graph": {
    "runudf1": {
      "process_id": "run_udf",
      "arguments": {
        "context": {},
        "data": [],
        "runtime": "Python",
        "udf": "\nfrom openeo.udf.udf_data import UdfData\nfrom openeo.udf.structured_data import StructuredData\n\ndef apply_udf_data(data: UdfData):\n    data.set_structured_data_list([\n        StructuredData(['s3://EODATA/Sentinel-2/MSI/L1C/2024/11/13/S2A_MSIL1C_20241113T101251_N0511_R022_T32TPQ_20241113T121135.SAFE'])\n    ])\n"
      },
      "result": true
    }
  }
}


In [5]:
cwl = Path("../cwl/force-l2.cwl").read_text()

In [9]:
force_pg = connection_cdse.datacube_from_process(
    "run_udf",
    data=None,
    udf=cwl,
    runtime="EOAP-CWL",
    context={"input": udf_process_graph},
)
force_fail_job = connection_cdse.create_job(force_pg, title="FORCE with query")
force_fail_job.start() 

Preflight process graph validation raised: [FeatureUnsupported] run_udf is not supported in validation mode.


<BatchJob job_id='j-26032614442447cca01e5f09420e3dd3'>

In [11]:
print(force_pg.to_json())

{
  "process_graph": {
    "runudf1": {
      "process_id": "run_udf",
      "arguments": {
        "context": {},
        "data": [],
        "runtime": "Python",
        "udf": "\nfrom openeo.udf.udf_data import UdfData\nfrom openeo.udf.structured_data import StructuredData\n\ndef apply_udf_data(data: UdfData):\n    data.set_structured_data_list([\n        StructuredData(['s3://EODATA/Sentinel-2/MSI/L1C/2024/11/13/S2A_MSIL1C_20241113T101251_N0511_R022_T32TPQ_20241113T121135.SAFE'])\n    ])\n"
      }
    },
    "runudf2": {
      "process_id": "run_udf",
      "arguments": {
        "context": {
          "input": {
            "from_node": "runudf1"
          }
        },
        "data": null,
        "runtime": "EOAP-CWL",
        "udf": "cwlVersion: v1.2\n\n# tested with\n# ssh yarn@archive03\n# cd integration/force/test3\n# . /home/yarn/opt/miniconda-cwltool/bin/activate\n# export AWS_ENDPOINT_URL_S3='https://eodata.dataspace.copernicus.eu'\n# export AWS_ACCESS_KEY_ID=...\n# expo